# 02 - Silver Produtos - Bike Store Medallion
Notebook de tratamento da camada Silver focado em produtos.

Fontes utilizadas da camada Bronze:
- `bike_store.bronze.products`
- `bike_store.bronze.categories`
- `bike_store.bronze.brands`
- `bike_store.bronze.stocks`

In [0]:
%run ./Utils/Qualidade_Dados

In [0]:
# Criando os dataframes de produtos e categorias
df_products   = spark.read.table("bike_store.bronze.products")
df_categories = spark.read.table("bike_store.bronze.categories")
df_brands     = spark.read.table("bike_store.bronze.brands")
df_stocks     = spark.read.table("bike_store.bronze.stocks")

In [0]:
# Analisando o esquema de df_products
df_products.printSchema()


In [0]:
# Mostra estatísticas básicas (count, mean, min, max, etc)
display(df_products.describe())

In [0]:
# Analisando o esquema de df_categories
df_categories.printSchema()

In [0]:
# Mostra estatísticas básicas (count, mean, min, max, etc)
display(df_categories.describe())

In [0]:
# Analisando o esquema de df_brands
df_brands.printSchema()

In [0]:
# Mostra estatísticas básicas (count, mean, min, max, etc)
display(df_brands.describe())

In [0]:
# Analisando o esquema de df_stocks
df_stocks.printSchema()

In [0]:
# Mostra estatísticas básicas (count, mean, min, max, etc)
display(df_stocks.describe())

In [0]:
# Verificando ocorrência de registros nulos por colunas
nulos_por_coluna(df_products, "products")
nulos_por_coluna(df_categories, "categories")
nulos_por_coluna(df_brands, "brands")
nulos_por_coluna(df_stocks, "stocks")

## Join Products + Categories + Brands


In [0]:
from pyspark.sql.functions import col

df_silver_products = df_products \
    .join(df_categories, on="category_id", how="left") \
    .join(df_brands, on="brand_id", how="left") \
    .select(
        col("product_id"),
        col("product_name"),
        col("category_name"),
        col("brand_name"),
        col("model_year"),
        col("list_price")
    )

display(df_silver_products)

## Agregar produtos da store por produtos e fazer o merge com produtos.

In [0]:
from pyspark.sql.functions import sum as spark_sum
# Agregando quantidades de produtos
df_total_stocks = df_stocks \
    .groupBy("product_id") \
    .agg(spark_sum("quantity").alias("total_quantity"))
display(df_total_stocks)


In [0]:
# Join com o dataframe de produtos enriquecido
df_silver_products_full = df_silver_products \
    .join(df_total_stocks, on="product_id", how="left")

display(df_silver_products_full)

In [0]:
# Salvando tabela de Produtos na camada Silver
df_silver_products_full.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("bike_store.silver.products")

print("✅ bike_store.silver.products salvo com sucesso!")